In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from IBL import IBL
from Parser import Parser
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.svm import SVC

from argument_parser import parse_arguments
from csv_writers import create_fw_k_ibl_csv_row, create_k_ibl_csv_row, create_ir_ibl_csv_row, create_svm_csv_row
from model_types import Models
from processing_types import (
    NormalizationStrategy, EncodingStrategy,
    MissingValuesNumericStrategy, MissingValuesCategoricalStrategy, RetentionPolicy
)

BASE_PATH = "../datasetsCBR/datasetsCBR/"
NUM_SPLITS = 1
RESULTS_PATH = "./results/"


if __name__ == "__main__":
    dataset_name = 'pen-based'
    
    ir_strategies = ["RENN"]

    parser = Parser(
        base_path=BASE_PATH,
        dataset_name=dataset_name,
        normalization_strategy=NormalizationStrategy.MEAN_NORMALIZE,
        encoding_strategy=EncodingStrategy.ONE_HOT_ENCODE, 
        missing_values_numeric_strategy=MissingValuesNumericStrategy.MEDIAN,
        missing_values_categorical_strategy=MissingValuesCategoricalStrategy.MODE,
        num_splits=NUM_SPLITS,
    )
    types = parser.get_types()
    post_encoding_types = parser.get_post_encoding_types()

    splits = [parser.get_split(fold) for fold in range(NUM_SPLITS)]

    all_labels = set()
    for tr, te in splits:
        all_labels.update(np.unique(tr.iloc[:, -1]))
        all_labels.update(np.unique(te.iloc[:, -1]))
    labels = np.array(sorted(all_labels))

    out_csv = Path(RESULTS_PATH + f"{dataset_name}-test.csv")
    # out_csv = Path(RESULTS_PATH + f"{dataset_name}-ir_k_ibl.csv")

    for ir_strategy in ir_strategies:

        rows = []
        for fold_id, (train_matrix, test_matrix) in enumerate(splits):

            t0 = time.perf_counter()

        
            ibl = IBL()

            # Fit + predict
            t0 = time.perf_counter()
            
            # --- Instance reduction strategy selection ---
            if ir_strategy == "IBL3":
                print("Applying IBL3 instance reduction...")
                ibl.fit(train_matrix, instance_red="IBL3")

            elif ir_strategy == "IBL3_verbose":
                print("Applying IBL3 (verbose) instance reduction...")
                ibl.fit(train_matrix, instance_red="IBL3_verbose")

            elif ir_strategy == "CNN":
                print("Applying Condensed Nearest Neighbor (CNN) instance reduction...")
                ibl.fit(train_matrix, instance_red="CNN")

            elif ir_strategy == "MCNN":
                print("Applying Modified Condensed Nearest Neighbor (MCNN) instance reduction...")
                ibl.fit(train_matrix, instance_red="MCNN")

            elif ir_strategy == "enn":
                print("Applying Edited Nearest Neighbor (ENN) instance reduction...")
                ibl.fit(train_matrix, instance_red="enn")

            elif ir_strategy == "RENN":
                print("Applying Repeated Edited Nearest Neighbor (RENN) instance reduction...")
                ibl.fit(train_matrix, instance_red="RENN")

            else:
                raise ValueError(f"Unknown instance reduction strategy")
        
            t1 = time.perf_counter()

           
            preds = ibl.run(
                test_matrix,
                k=7,
                metric='cosine',
                vote='borda',
                retention_policy=RetentionPolicy.ALWAYS_RETAIN,
                types=types,
            )

            t2 = time.perf_counter()

            # Times
            fit_time = t1 - t0
            predict_time = t2 - t1
            total_time = t2 - t0

            # Metrics
            y_true = test_matrix.iloc[:, -1].to_numpy()
            y_pred = np.asarray(preds)

            acc = accuracy_score(y_true, y_pred)

            pM, rM, fM, _ = precision_recall_fscore_support(
                y_true, y_pred, average="macro", zero_division=0
            )
            pW, rW, fW, _ = precision_recall_fscore_support(
                y_true, y_pred, average="weighted", zero_division=0
            )

            cm_fold = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

     
            row = create_ir_ibl_csv_row(
                metric='cosine',
                k=7,
                vote='borda',
                retention=RetentionPolicy.ALWAYS_RETAIN,
                fold_id=fold_id,
                num_folds=NUM_SPLITS,
                n_train=train_matrix.shape[0],
                n_test=test_matrix.shape[0],
                fit_time=fit_time,
                predict_time=predict_time,
                total_time=total_time,
                accuracy=acc,
                precision_macro=pM,
                recall_macro=rM,
                f1_macro=fM,
                precision_weighted=pW,
                recall_weighted=rW,
                f1_weighted=fW,
                confusion_matrix=cm_fold,
                labels=labels,
                instance_reduction_method=ir_strategy,
                memory_before_ir=ibl.cp_before_ir,
                memory_after_ir=ibl.cp_after_ir,
                memory_after_training=ibl.cp_after_training,
                percentage_memory_reduction=(ibl.cp_before_ir - ibl.cp_after_ir) / ibl.cp_before_ir * 100
            )
            rows.append(row)

        out_csv.parent.mkdir(parents=True, exist_ok=True)
        df_rows = pd.DataFrame(rows)

        write_header = not out_csv.exists()
        df_rows.to_csv(out_csv, mode="a", header=write_header, index=False)



Applying Repeated Edited Nearest Neighbor (RENN) instance reduction...
Instances before: 9894
Instances after: 9829 (99.34% retained)
Instances before: 9829
Instances after: 9825 (99.96% retained)
Instances before: 9825
Instances after: 9824 (99.99% retained)
Instances before: 9824
Instances after: 9823 (99.99% retained)
Instances before: 9823
Instances after: 9822 (99.99% retained)
Instances before: 9822
Instances after: 9821 (99.99% retained)
Instances before: 9821
Instances after: 9821 (100.0% retained)
No more instances removed → stopping.
Instances before: 9894
Instances after RENN: 9821 (99.26% retained)
Instances reduced from 9894 to 9821

Storage used with specified matrix: 1.28 MB
Storage used by X: 1.20 MB
Preallocating matrix of shape (10919, 16)
Total time for all instances: 0.23s

Storage used with specified matrix: 1.33 MB
Final training set size: (10919, 16)
